In [1]:
# Standard library imports
import os
from typing import List, Tuple, Dict
import pickle

# Third-party imports
import numpy as np
import wfdb
from scipy.signal import butter, filtfilt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
import tensorflow as tf
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Conv1D, LSTM, Dense, Dropout, BatchNormalization # type: ignore
from tensorflow.keras.optimizers import Adam # type: ignore
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint # type: ignore
import matplotlib.pyplot as plt

MITDB_PATH = '/Users/sidpatsamatla/Desktop/CardioBound_Code/CardioBound_Data/mitdb'
AFDB_PATH = '/Users/sidpatsamatla/Desktop/CardioBound_Code/CardioBound_Data/afdb'
PTBDB_PATH = '/Users/sidpatsamatla/Desktop/CardioBound_Code/CardioBound_Data/ptbdb'
LTAFDB_PATH = '/Users/sidpatsamatla/Desktop/CardioBound_Code/CardioBound_Data/ltafdb'
MODEL_SAVE_PATH = '/Users/sidpatsamatla/Desktop/CardioBound_Code/models/'
RESULTS_PATH = '/Users/sidpatsamatla/Desktop/CardioBound_Code/results/'

SAMPLING_FREQUENCY = 360
LOWCUT_FREQ = 0.5
HIGHCUT_FREQ = 40.0
FILTER_ORDER = 4

WINDOW_SIZE = 360
OVERLAP = 180

NUM_CLASSES = 5
CNN_FILTERS_1 = 64
CNN_FILTERS_2 = 128
LSTM_UNITS_1 = 128
LSTM_UNITS_2 = 64
DENSE_UNITS = 64
DROPOUT_RATE = 0.3

BATCH_SIZE = 32
EPOCHS = 10  # Reduced for testing
LEARNING_RATE = 0.001

RISK_CLASSES = 3
RISK_FEATURES = ['mean_rr', 'std_rr', 'rmssd', 'mean_hr', 'arrhythmia_count']

MODEL_CHECKPOINT_INTERVAL = 5
EARLY_STOPPING_PATIENCE = 10

MITDB_RECORDS = list(range(100, 103))  
AFDB_RECORDS = ['04015']                
LTAFDB_RECORDS = ['00', '01', '02', '03', '04']               

ARRHYTHMIA_MAPPING = {
    'N': 0, 'L': 0, 'R': 0, 'e': 0, 'j': 0,
    'A': 1, 'a': 1, 'S': 1, 'J': 1,
    'V': 2, 'E': 2,
    'F': 3,
    '/': 4, 'f': 4, 'Q': 4
}

print("✓ Configuration loaded")

✓ Configuration loaded


In [ ]:
def download_all_datasets():
    base_path = '/Users/sidpatsamatla/Desktop/CardioBound_Code/CardioBound_Data'
    os.makedirs(base_path, exist_ok=True)
    
    print("Downloading datasets (this may take several minutes)...")
    print("-" * 60)
    
    try:
        print("\n[1/4] Downloading MIT-BIH Arrhythmia Database...")
        wfdb.dl_database('mitdb', dl_dir=f'{base_path}/mitdb')
        print("✓ MIT-BIH downloaded successfully")
    except Exception as e:
        print(f"✗ Failed to download MIT-BIH: {e}")
    
    try:
        print("\n[2/4] Downloading MIT-BIH Atrial Fibrillation Database...")
        wfdb.dl_database('afdb', dl_dir=f'{base_path}/afdb')
        print("✓ AFDB downloaded successfully")
    except Exception as e:
        print(f"✗ Failed to download AFDB: {e}")
    
    try:
        print("\n[3/4] Downloading Long-Term AF Database...")
        wfdb.dl_database('ltafdb', dl_dir=f'{base_path}/ltafdb')
        print("✓ LTAFDB downloaded successfully")
    except Exception as e:
        print(f"✗ Failed to download LTAFDB: {e}")
    
    try:
        print("\n[4/4] Downloading PTB Diagnostic Database...")
        wfdb.dl_database('ptbdb', dl_dir=f'{base_path}/ptbdb')
        print("✓ PTBDB downloaded successfully")
    except Exception as e:
        print(f"✗ Failed to download PTBDB: {e}")
    
    print("\n" + "=" * 60)
    print("All datasets downloaded!")
    print("=" * 60)

# Run the download
download_all_datasets()

Hello


In [2]:
def load_mitdb_records(data_path: str, record_numbers: List[int]) -> Tuple[List[np.ndarray], List[List[str]]]:
    signals = []
    annotations = []
    data_path = data_path.rstrip('/')
    
    for record_num in record_numbers:
        try:
            record = wfdb.rdrecord(f'{data_path}/{record_num}')
            annotation = wfdb.rdann(f'{data_path}/{record_num}', 'atr')
            
            signals.append(record.p_signal[:, 0])
            annotations.append(annotation.symbol)
            
            print(f"✓ Loaded MITDB record {record_num}")
        except Exception as e:
            print(f"✗ Failed MITDB record {record_num}: {e}")
    
    return signals, annotations


def load_afdb_records(data_path: str, record_numbers: List[str]) -> Tuple[List[np.ndarray], List[List[str]]]:
    signals = []
    annotations = []
    data_path = data_path.rstrip('/')
    
    for record_num in record_numbers:
        try:
            record = wfdb.rdrecord(f'{data_path}/{record_num}')
            annotation = wfdb.rdann(f'{data_path}/{record_num}', 'atr')
            
            signals.append(record.p_signal[:, 0])
            annotations.append(annotation.symbol)
            
            print(f"✓ Loaded AFDB record {record_num}")
        except Exception as e:
            print(f"✗ Failed AFDB record {record_num}: {e}")
    
    return signals, annotations


def load_ltafdb_records(data_path: str, record_numbers: List[str]) -> Tuple[List[np.ndarray], List[List[str]]]:
    signals = []
    annotations = []
    data_path = data_path.rstrip('/')
    
    for record_num in record_numbers:
        try:
            record = wfdb.rdrecord(f'{data_path}/{record_num}')
            annotation = wfdb.rdann(f'{data_path}/{record_num}', 'atr')
            
            signals.append(record.p_signal[:, 0])
            annotations.append(annotation.symbol)
            
            print(f"✓ Loaded LTAFDB record {record_num}")
        except Exception as e:
            print(f"✗ Failed LTAFDB record {record_num}: {e}")
    
    return signals, annotations

def load_all_datasets() -> Tuple[List[np.ndarray], List[List[str]]]:
    """Load data from three datasets and combine them."""
    all_signals = []
    all_annotations = []
    
    print("\n[Loading MIT-BIH Database]")
    mitdb_signals, mitdb_annotations = load_mitdb_records(MITDB_PATH, MITDB_RECORDS)
    all_signals.extend(mitdb_signals)
    all_annotations.extend(mitdb_annotations)
    
    print("\n[Loading AFDB Database]")
    afdb_signals, afdb_annotations = load_afdb_records(AFDB_PATH, AFDB_RECORDS)
    all_signals.extend(afdb_signals)
    all_annotations.extend(afdb_annotations)
    
    print("\n[Loading LTAFDB Database]")
    ltafdb_signals, ltafdb_annotations = load_ltafdb_records(LTAFDB_PATH, LTAFDB_RECORDS)
    all_signals.extend(ltafdb_signals)
    all_annotations.extend(ltafdb_annotations)
    
    # SKIP PTBDB - took too long to download
    # print("\n[Loading PTBDB Database]")
    # ptbdb_signals, ptbdb_annotations = load_ptbdb_records(PTBDB_PATH, PTBDB_RECORDS)
    # all_signals.extend(ptbdb_signals)
    # all_annotations.extend(ptbdb_annotations)
    
    print(f"\nTotal signals loaded across all databases: {len(all_signals)}")
    print(f"Total annotations loaded: {len(all_annotations)}")
    
    return all_signals, all_annotations

print("✓ All data loading functions defined")

✓ All data loading functions defined


In [3]:
def bandpass_filter(signal: np.ndarray, lowcut: float = LOWCUT_FREQ, highcut: float = HIGHCUT_FREQ, 
                   fs: int = SAMPLING_FREQUENCY, order: int = FILTER_ORDER) -> np.ndarray:
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    filtered_signal = filtfilt(b, a, signal)
    return filtered_signal


def segment_signal(signal: np.ndarray, window_size: int = WINDOW_SIZE, overlap: int = OVERLAP) -> np.ndarray:
    segments = []
    step = window_size - overlap
    
    for i in range(0, len(signal) - window_size, step):
        segment = signal[i:i + window_size]
        segments.append(segment)
    
    return np.array(segments)


def normalize_signal(signal: np.ndarray) -> np.ndarray:
    mean = np.mean(signal)
    std = np.std(signal)
    normalized = (signal - mean) / (std + 1e-8)
    return normalized

print("✓ Signal processing functions defined")

✓ Signal processing functions defined


In [4]:
def detect_r_peaks(signal: np.ndarray, fs: int = SAMPLING_FREQUENCY) -> np.ndarray:
    threshold = 0.6 * np.max(signal)
    peaks = []
    
    for i in range(1, len(signal) - 1):
        if signal[i] > threshold and signal[i] > signal[i-1] and signal[i] > signal[i+1]:
            peaks.append(i)
    
    return np.array(peaks)


def extract_hrv_features(r_peaks: np.ndarray, fs: int = SAMPLING_FREQUENCY) -> Dict[str, float]:
    if len(r_peaks) < 2:
        return {'mean_rr': 0, 'std_rr': 0, 'rmssd': 0, 'mean_hr': 0}
    
    rr_intervals = np.diff(r_peaks) / fs * 1000
    
    if len(rr_intervals) == 0:
        return {'mean_rr': 0, 'std_rr': 0, 'rmssd': 0, 'mean_hr': 0}
    
    features = {
        'mean_rr': np.mean(rr_intervals),
        'std_rr': np.std(rr_intervals),
        'rmssd': np.sqrt(np.mean(np.square(np.diff(rr_intervals)))) if len(rr_intervals) > 1 else 0,
        'mean_hr': 60000 / np.mean(rr_intervals) if np.mean(rr_intervals) > 0 else 0
    }
    
    return features


def extract_hrv_from_signals(signals: List[np.ndarray]) -> np.ndarray:
    all_features = []
    
    for signal in signals:
        filtered = bandpass_filter(signal)
        r_peaks = detect_r_peaks(filtered)
        hrv_features = extract_hrv_features(r_peaks)
        
        feature_vector = [
            hrv_features['mean_rr'],
            hrv_features['std_rr'],
            hrv_features['rmssd'],
            hrv_features['mean_hr']
        ]
        
        all_features.append(feature_vector)
    
    return np.array(all_features)

print("✓ Feature extraction functions defined")

✓ Feature extraction functions defined


In [5]:
def map_labels_to_classes(annotations: List[str]) -> List[int]:
    mapped = []
    for label in annotations:
        if label in ARRHYTHMIA_MAPPING:
            mapped.append(ARRHYTHMIA_MAPPING[label])
        else:
            mapped.append(4)
    return mapped


def prepare_dataset(signals: List[np.ndarray], annotations: List[List[str]]) -> Tuple[np.ndarray, np.ndarray]:
    X_all = []
    y_all = []
    
    for signal, annotation in zip(signals, annotations):
        filtered = bandpass_filter(signal)
        segments = segment_signal(filtered)
        
        samples_per_segment = WINDOW_SIZE - OVERLAP
        
        for i, segment in enumerate(segments):
            start_sample = i * samples_per_segment
            
            if len(annotation) > 0:
                label_idx = min(int(start_sample / 360), len(annotation) - 1)
                label = annotation[label_idx] if label_idx < len(annotation) else 'N'
                mapped_label = ARRHYTHMIA_MAPPING.get(label, 4)
                
                X_all.append(segment)
                y_all.append(mapped_label)  # This is now an integer
    
    X_all = np.array(X_all)
    y_all = np.array(y_all, dtype=np.int32)  # ← FIX: Force integer type
    
    print(f"Total segments created: {len(X_all)}")
    print(f"Total labels: {len(y_all)}")
    
    # Check if y_all has any values before bincount
    if len(y_all) > 0:
        print(f"Label distribution: {np.bincount(y_all)}")
    else:
        print("No labels found!")
    
    return X_all, y_all


def split_and_normalize(X: np.ndarray, y: np.ndarray) -> Tuple:
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
    
    train_mean = np.mean(X_train, axis=0)
    train_std = np.std(X_train, axis=0)
    
    X_train_norm = (X_train - train_mean) / (train_std + 1e-8)
    X_val_norm = (X_val - train_mean) / (train_std + 1e-8)
    X_test_norm = (X_test - train_mean) / (train_std + 1e-8)
    
    return X_train_norm, X_val_norm, X_test_norm, y_train, y_val, y_test

print("✓ Data preparation functions defined (FIXED)")

✓ Data preparation functions defined (FIXED)


In [6]:
def build_cnn_lstm_model(input_shape: Tuple[int, int], num_classes: int = NUM_CLASSES) -> Sequential:
    model = Sequential([
        Conv1D(CNN_FILTERS_1, kernel_size=5, activation='relu', input_shape=input_shape),
        BatchNormalization(),
        Conv1D(CNN_FILTERS_2, kernel_size=5, activation='relu'),
        BatchNormalization(),
        Dropout(DROPOUT_RATE),
        
        LSTM(LSTM_UNITS_1, return_sequences=True),
        Dropout(DROPOUT_RATE),
        LSTM(LSTM_UNITS_2, return_sequences=False),
        Dropout(DROPOUT_RATE),
        
        Dense(DENSE_UNITS, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model


def build_risk_stratification_model() -> LogisticRegression:
    model = LogisticRegression(max_iter=1000, random_state=42, multi_class='multinomial')
    return model

print("✓ Model architectures defined")

✓ Model architectures defined


In [7]:
def train_classification_model(model: Sequential, X_train: np.ndarray, y_train: np.ndarray, 
                               X_val: np.ndarray, y_val: np.ndarray) -> Tuple:
    os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
    
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True),
        ModelCheckpoint(f'{MODEL_SAVE_PATH}/best_model.keras', monitor='val_accuracy', save_best_only=True)
    ]
    
    history = model.fit(
        X_train, y_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_val, y_val),
        callbacks=callbacks,
        verbose=1
    )
    
    return model, history


def train_risk_model(signals: List[np.ndarray], annotations: List[List[str]]) -> LogisticRegression:
    X_hrv = extract_hrv_from_signals(signals)
    
    y_risk = []
    for annotation in annotations:
        mapped_labels = map_labels_to_classes(annotation)
        arrhythmia_count = sum(1 for label in mapped_labels if label != 0)
        total_beats = len(mapped_labels)
        
        arrhythmia_ratio = arrhythmia_count / total_beats if total_beats > 0 else 0
        
        if arrhythmia_ratio < 0.1:
            risk = 0
        elif arrhythmia_ratio < 0.3:
            risk = 1
        else:
            risk = 2
        
        y_risk.append(risk)
    
    y_risk = np.array(y_risk)
    
    X_train, X_test, y_train, y_test = train_test_split(X_hrv, y_risk, test_size=0.2, random_state=42)
    
    model = build_risk_stratification_model()
    model.fit(X_train, y_train)
    
    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)
    
    print(f"\nRisk Stratification Model:")
    print(f"Training Accuracy: {train_score:.4f}")
    print(f"Test Accuracy: {test_score:.4f}")
    
    return model

print("✓ Training functions defined")

✓ Training functions defined


In [8]:
def evaluate_model(model: Sequential, X_test: np.ndarray, y_test: np.ndarray) -> None:
    os.makedirs(RESULTS_PATH, exist_ok=True)
    
    y_pred = np.argmax(model.predict(X_test), axis=1)
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, 
                               target_names=['Normal', 'Supraventricular', 'Ventricular', 'Fusion', 'Unknown']))
    
    cm = confusion_matrix(y_test, y_pred)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Normal', 'Supraventricular', 'Ventricular', 'Fusion', 'Unknown'],
                yticklabels=['Normal', 'Supraventricular', 'Ventricular', 'Fusion', 'Unknown'])
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'{RESULTS_PATH}/confusion_matrix.png')
    print(f"Confusion matrix saved to {RESULTS_PATH}/confusion_matrix.png")


def plot_training_history(history) -> None:
    os.makedirs(RESULTS_PATH, exist_ok=True)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    ax1.plot(history.history['accuracy'], label='Train Accuracy')
    ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
    ax1.set_title('Model Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True)
    
    ax2.plot(history.history['loss'], label='Train Loss')
    ax2.plot(history.history['val_loss'], label='Val Loss')
    ax2.set_title('Model Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.savefig(f'{RESULTS_PATH}/training_history.png')
    print(f"Training history saved to {RESULTS_PATH}/training_history.png")


def save_models(classification_model: Sequential, risk_model: LogisticRegression) -> None:
    os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
    
    classification_model.save(f'{MODEL_SAVE_PATH}/classification_model.keras')
    print(f"Classification model saved to {MODEL_SAVE_PATH}/classification_model.keras")
    
    with open(f'{MODEL_SAVE_PATH}/risk_model.pkl', 'wb') as f:
        pickle.dump(risk_model, f)
    print(f"Risk stratification model saved to {MODEL_SAVE_PATH}/risk_model.pkl")

    print("✓ Evaluation functions defined")

In [20]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import os


def evaluate_model(model, X_test, y_test, save_path="results") -> None:
    """
    Evaluate classification model:
    - Classification report
    - Confusion matrix
    """

    os.makedirs(save_path, exist_ok=True)

    # ===== Predict =====
    y_pred_probs = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # ===== Class names MUST match model output =====
    CLASS_NAMES = [
        "Normal",
        "Supraventricular",
        "Ventricular",
        "Fusion",
        "Unknown"
    ]

    num_classes = model.output_shape[-1]

    # ===== Hard safety check =====
    assert num_classes == len(CLASS_NAMES), (
        f"Model outputs {num_classes} classes, "
        f"but {len(CLASS_NAMES)} class names were provided."
    )

    # ===== Classification report =====
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        labels=list(range(num_classes)),
        target_names=CLASS_NAMES,
        zero_division=0
    ))

    # ===== Confusion matrix =====
    cm = confusion_matrix(y_test, y_pred, labels=list(range(num_classes)))

    plt.figure(figsize=(10, 8))
    plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    plt.title("Confusion Matrix", fontsize=14, fontweight="bold")
    plt.colorbar()

    tick_marks = np.arange(num_classes)
    plt.xticks(tick_marks, CLASS_NAMES, rotation=45)
    plt.yticks(tick_marks, CLASS_NAMES)

    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(
                j, i, format(cm[i, j], "d"),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black"
            )

    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()

    save_file = os.path.join(save_path, "confusion_matrix.png")
    plt.savefig(save_file, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"✓ Confusion matrix saved to {save_file}")



In [10]:
def export_for_hardware():
    print("\n" + "=" * 60)
    print("Exporting models for hardware deployment")
    print("=" * 60)
    os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
    
    model = tf.keras.models.load_model(f'{MODEL_SAVE_PATH}/classification_model.keras')
    
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    
    # Fix for LSTM models: Include TensorFlow ops and disable tensor list lowering
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,  # Enable TFLite ops
        tf.lite.OpsSet.SELECT_TF_OPS      # Enable select TensorFlow ops
    ]
    converter._experimental_lower_tensor_list_ops = False
    
    tflite_model = converter.convert()
    
    with open(f'{MODEL_SAVE_PATH}/model.tflite', 'wb') as f:
        f.write(tflite_model)
    
    print(f"✓ TensorFlow Lite model saved to {MODEL_SAVE_PATH}/model.tflite")
    print("\nTo deploy to Raspberry Pi:")
    print(f"  scp {MODEL_SAVE_PATH}/model.tflite pi@raspberrypi.local:/home/pi/cardiobound/")
    print("\nTo deploy to Jetson Nano:")
    print(f"  scp {MODEL_SAVE_PATH}/model.tflite nvidia@jetson.local:/home/nvidia/cardiobound/")
    print("✓ Hardware export function defined")
print("✓ Hardware export function defined")

✓ Hardware export function defined


In [11]:
print("=" * 60)
print("CardioBound - AI-Powered Arrhythmia Detection System")
print("=" * 60)

print("\n[PHASE 1: ARRHYTHMIA CLASSIFICATION MODEL]")
print("-" * 60)

print("\n[1/8] Loading MIT-BIH data...")
signals, annotations = load_mitdb_records(MITDB_PATH, MITDB_RECORDS)

print(f"\nNumber of signals loaded: {len(signals)}")
print(f"Number of annotations loaded: {len(annotations)}")

if len(signals) == 0:
    print("ERROR: No data loaded. Check your MITDB_PATH and ensure files exist.")
else:
    print("✓ Data loaded successfully")

CardioBound - AI-Powered Arrhythmia Detection System

[PHASE 1: ARRHYTHMIA CLASSIFICATION MODEL]
------------------------------------------------------------

[1/8] Loading MIT-BIH data...
✓ Loaded MITDB record 100
✓ Loaded MITDB record 101
✓ Loaded MITDB record 102

Number of signals loaded: 3
Number of annotations loaded: 3
✓ Data loaded successfully


In [12]:
print("\n[2/8] Preparing dataset...")
X, y = prepare_dataset(signals, annotations)
print(f"Dataset shape: {X.shape}")

print("\n[3/8] Splitting and normalizing...")
X_train, X_val, X_test, y_train, y_val, y_test = split_and_normalize(X, y)

X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_val = X_val.reshape(X_val.shape[0], X_val.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(f"Train shape: {X_train.shape}")
print(f"Val shape: {X_val.shape}")
print(f"Test shape: {X_test.shape}")


[2/8] Preparing dataset...
Total segments created: 10830
Total labels: 10830
Label distribution: [7336   56    6    0 3432]
Dataset shape: (10830, 360)

[3/8] Splitting and normalizing...
Train shape: (7581, 360, 1)
Val shape: (1624, 360, 1)
Test shape: (1625, 360, 1)


In [13]:
print("\n[4/8] Building CNN-LSTM model...")
classification_model = build_cnn_lstm_model(input_shape=(WINDOW_SIZE, 1))
classification_model.summary()

print("\n[5/8] Training classification model...")
classification_model, history = train_classification_model(classification_model, X_train, y_train, X_val, y_val)


[4/8] Building CNN-LSTM model...


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 356, 64)        │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 356, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 352, 128)       │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 352, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 352, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 352, 128)       │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 352, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 227,717 (889.52 KB)

 Trainable params: 227,333 (888.02 KB)

 Non-trainable params: 384 (1.50 KB)


[5/8] Training classification model...
Epoch 1/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 43s 175ms/step - accuracy: 0.7124 - loss: 0.6869 - val_accuracy: 0.3824 - val_loss: 1.0073
Epoch 2/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 40s 167ms/step - accuracy: 0.9624 - loss: 0.1843 - val_accuracy: 0.9563 - val_loss: 0.1565
Epoch 3/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 41s 172ms/step - accuracy: 0.9687 - loss: 0.1651 - val_accuracy: 0.9692 - val_loss: 0.1368
Epoch 4/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 40s 167ms/step - accuracy: 0.8668 - loss: 0.3816 - val_accuracy: 0.9600 - val_loss: 0.1945
Epoch 5/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 41s 172ms/step - accuracy: 0.9621 - loss: 0.1856 - val_accuracy: 0.9544 - val_loss: 0.1575
Epoch 6/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 39s 166ms/step - accuracy: 0.9641 - loss: 0.1637 - val_accuracy: 0.9711 - val_loss: 0.1226
Epoch 7/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 41s 173ms/step - accuracy: 0.9731 - loss: 0.1335 - val_accuracy: 0.9698 - val_loss: 0.1207
Epoch 8/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 40s

In [14]:
print("\n[6/8] Evaluating classification model...")
test_loss, test_acc = classification_model.evaluate(X_test, y_test)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}%")


[6/8] Evaluating classification model...
51/51 ━━━━━━━━━━━━━━━━━━━━ 3s 54ms/step - accuracy: 0.9742 - loss: 0.1012

Test Loss: 0.1012
Test Accuracy: 0.9742%


In [21]:
print_training_summary(history)
evaluate_model(classification_model, X_test, y_test)
plot_training_history(history)

print("\n✓ All visualizations saved to results folder!")


TRAINING SUMMARY

Final Results (Epoch 10):
  Training Accuracy:   0.9734
  Validation Accuracy: 0.9748
  Training Loss:       0.1213
  Validation Loss:     0.1151

Best Validation Accuracy: 0.9754 (Epoch 9)
Best Validation Loss:     0.1126 (Epoch 9)

Overfitting Analysis:
  Accuracy Gap: -0.0014 ✓ Good generalization
  Loss Gap:     -0.0063 ✓ Good generalization


Classification Report:
                  precision    recall  f1-score   support

          Normal       0.98      0.98      0.98      1101
Supraventricular       0.00      0.00      0.00         8
     Ventricular       0.00      0.00      0.00         1
          Fusion       0.00      0.00      0.00         0
         Unknown       0.96      0.97      0.97       515

        accuracy                           0.97      1625
       macro avg       0.39      0.39      0.39      1625
    weighted avg       0.97      0.97      0.97      1625

✓ Confusion matrix saved to results/confusion_matrix.png
✓ Comprehensive training h

In [16]:
print("\n[PHASE 2: RISK STRATIFICATION MODEL]")
print("-" * 60)

print("\n[7/8] Training risk stratification model...")
risk_model = train_risk_model(signals, annotations)


[PHASE 2: RISK STRATIFICATION MODEL]
------------------------------------------------------------

[7/8] Training risk stratification model...

Risk Stratification Model:
Training Accuracy: 1.0000
Test Accuracy: 0.0000


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1262: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.
  warnings.warn(


In [17]:
print("\n[8/8] Saving models...")
save_models(classification_model, risk_model)

print("\n" + "=" * 60)
print("PHASE 1 & 2 COMPLETE!")
print("=" * 60)
print("\nModels saved:")
print(f"  - Classification: {MODEL_SAVE_PATH}/classification_model.keras")
print(f"  - Risk Stratification: {MODEL_SAVE_PATH}/risk_model.pkl")
print(f"\nResults saved to: {RESULTS_PATH}/")


[8/8] Saving models...
Classification model saved to /Users/sidpatsamatla/Desktop/CardioBound_Code/models//classification_model.keras
Risk stratification model saved to /Users/sidpatsamatla/Desktop/CardioBound_Code/models//risk_model.pkl
✓ Evaluation functions defined

PHASE 1 & 2 COMPLETE!

Models saved:
  - Classification: /Users/sidpatsamatla/Desktop/CardioBound_Code/models//classification_model.keras
  - Risk Stratification: /Users/sidpatsamatla/Desktop/CardioBound_Code/models//risk_model.pkl

Results saved to: /Users/sidpatsamatla/Desktop/CardioBound_Code/results//


In [17]:
# Run this chunk if you want to export for hardware deployment
export_for_hardware()


Exporting models for hardware deployment
INFO:tensorflow:Assets written to: /var/folders/lq/yzhlrqc1097bbqh1vkh2s4b80000gn/T/tmpwzyq8azd/assets


INFO:tensorflow:Assets written to: /var/folders/lq/yzhlrqc1097bbqh1vkh2s4b80000gn/T/tmpwzyq8azd/assets


Saved artifact at '/var/folders/lq/yzhlrqc1097bbqh1vkh2s4b80000gn/T/tmpwzyq8azd'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 360, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  6357440976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  6357444432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  6357443472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  6357444240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  6357442704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  6357444048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  6348550608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  6357442320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  6350237200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  6350236624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  6357445584: TensorSpec(sha

W0000 00:00:1763786881.681710  173308 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1763786881.681724  173308 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2025-11-21 23:48:01.682061: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/lq/yzhlrqc1097bbqh1vkh2s4b80000gn/T/tmpwzyq8azd
2025-11-21 23:48:01.682694: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2025-11-21 23:48:01.682698: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/lq/yzhlrqc1097bbqh1vkh2s4b80000gn/T/tmpwzyq8azd
I0000 00:00:1763786881.689726  173308 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2025-11-21 23:48:01.690804: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2025-11-21 23:48:01.729947: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /var/folder